In [2]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

In [13]:
# Generate 1000 rows of (5, 2000) data
# Set seed for reproducibility
np.random.seed(42)

# Generate random data with 1000 rows and 10000 features
data = np.random.randint(0, 2, size=(1000, 20000))
label = np.random.randint(0, 2, size=(1000, 1))

# Reshape each row to have a 5x2000 structure
reshaped_data = data.reshape(1000, 5, 4000)

In [17]:
print(label.shape)
print(reshaped_data.shape)

(1000, 1)
(1000, 5, 4000)


In [19]:
# Convert to tensor
X = torch.from_numpy(reshaped_data).float().unsqueeze(1)
print(X)
print(X.shape)

tensor([[[[0., 1., 0.,  ..., 1., 1., 1.],
          [1., 0., 0.,  ..., 1., 0., 0.],
          [1., 0., 1.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 1., 1., 0.],
          [1., 1., 1.,  ..., 0., 1., 0.]]],


        [[[0., 1., 1.,  ..., 1., 0., 1.],
          [1., 1., 0.,  ..., 1., 1., 1.],
          [1., 1., 0.,  ..., 0., 0., 1.],
          [0., 0., 0.,  ..., 1., 1., 1.],
          [1., 1., 0.,  ..., 0., 1., 1.]]],


        [[[1., 1., 1.,  ..., 0., 1., 0.],
          [1., 0., 0.,  ..., 0., 1., 1.],
          [1., 1., 1.,  ..., 1., 0., 1.],
          [0., 1., 0.,  ..., 0., 1., 0.],
          [1., 0., 1.,  ..., 1., 0., 0.]]],


        ...,


        [[[1., 0., 0.,  ..., 0., 0., 0.],
          [0., 1., 1.,  ..., 1., 0., 1.],
          [0., 0., 1.,  ..., 0., 0., 1.],
          [1., 0., 1.,  ..., 1., 0., 0.],
          [1., 1., 1.,  ..., 0., 0., 0.]]],


        [[[1., 1., 1.,  ..., 1., 0., 1.],
          [0., 0., 1.,  ..., 0., 1., 0.],
          [1., 1., 1.,  ..., 1., 1., 0.],
   

In [20]:
input_dim = [5, 4000]
out_channels = 50
conv_kernel_size = 10
max_pooling_kernel_size = 5

In [26]:
conv = nn.Conv2d(1, out_channels, kernel_size=(input_dim[0], conv_kernel_size))
relu = nn.ReLU()
pool = nn.MaxPool2d((1, max_pooling_kernel_size))
flatten = nn.Flatten()

In [30]:
flatten_dim = out_channels * ((input_dim[1] - conv_kernel_size + 1) // max_pooling_kernel_size)
flatten_dim

39900

In [31]:
linear1 = nn.Linear(flatten_dim, 625)
linear2 = nn.Linear(625, 125)
linear3 = nn.Linear(125, 2)

In [39]:
softmax = nn.LogSoftmax(dim=1)

In [48]:
print(X.shape)

x = conv(X)

torch.Size([1000, 1, 5, 4000])


In [49]:
print(x.shape)
print(x[0, 1])

torch.Size([1000, 50, 1, 3991])
tensor([[-0.1736,  0.0689, -0.3150,  ..., -0.1209, -0.2246, -0.8681]],
       grad_fn=<SelectBackward0>)


In [50]:
x = relu(x)
print(x.shape)

x = pool(x)
print(x.shape)

x = flatten(x)
print(x.shape)

torch.Size([1000, 50, 1, 3991])
torch.Size([1000, 50, 1, 798])
torch.Size([1000, 39900])


In [36]:
x = linear1(x)
print(x.shape)
print(x)

torch.Size([1000, 625])
tensor([[-0.0744,  0.2333, -0.2633,  ...,  0.2507, -0.1604,  0.2433],
        [-0.0962,  0.1351, -0.0566,  ...,  0.2848, -0.0097,  0.3141],
        [ 0.0953,  0.2562, -0.2592,  ...,  0.3666, -0.1146,  0.4437],
        ...,
        [-0.0993,  0.2750, -0.1768,  ...,  0.2996, -0.0564,  0.4765],
        [-0.2522,  0.2020, -0.1067,  ...,  0.5820, -0.0799,  0.3695],
        [-0.1943,  0.2203, -0.0702,  ...,  0.5221, -0.0644,  0.6028]],
       grad_fn=<AddmmBackward0>)


In [37]:
x = linear2(x)
print(x.shape)
print(x)

torch.Size([1000, 125])
tensor([[ 0.0849, -0.0215, -0.0589,  ..., -0.0321, -0.0268,  0.1080],
        [ 0.1427, -0.1016,  0.0408,  ..., -0.0777,  0.0063, -0.0386],
        [ 0.1891, -0.0199,  0.0335,  ..., -0.0414, -0.0899,  0.0048],
        ...,
        [ 0.1146, -0.0099, -0.0043,  ..., -0.0306, -0.0969, -0.0637],
        [ 0.1508,  0.0542,  0.0589,  ..., -0.1418, -0.1478, -0.0182],
        [ 0.1204, -0.0326,  0.0913,  ..., -0.1398, -0.0154, -0.0529]],
       grad_fn=<AddmmBackward0>)


In [38]:
x = linear3(x)
print(x.shape)
print(x)

torch.Size([1000, 2])
tensor([[-0.1299,  0.0584],
        [-0.1226,  0.0397],
        [-0.2000,  0.0820],
        ...,
        [-0.1460, -0.0213],
        [-0.1653,  0.0151],
        [-0.1592,  0.0064]], grad_fn=<AddmmBackward0>)


In [40]:
y_pred = softmax(x)
print(y_pred.shape)
print(y_pred)

torch.Size([1000, 2])
tensor([[-0.7917, -0.6034],
        [-0.7776, -0.6153],
        [-0.8441, -0.5620],
        ...,
        [-0.7574, -0.6328],
        [-0.7875, -0.6070],
        [-0.7793, -0.6138]], grad_fn=<LogSoftmaxBackward0>)
